# 🚀 YOLO ONNX Cloud Trainer (Google Colab - Free GPU)
### Fast GPU-Accelerated Training with Zero-Data Retention Policy

This notebook trains YOLO object detection models from any dataset JSON directly on a **free NVIDIA GPU (Tesla T4)** via Google Colab.

> **🔒 Privacy & Security:** All images and labels are decoded into temporary virtual machine memory and completely purged upon exporting the `.onnx` model.

In [ ]:
# 1. Install Computer Vision & Neural Network Libraries
!pip install -q ultralytics onnx onnxslim

In [ ]:
# 2. Upload Dataset JSON File (e.g. dataset.json)
from google.colab import files
import os
import glob

print("Please select your dataset .json file:")
uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f"✓ Dataset successfully loaded: {dataset_filename}")

In [ ]:
# 3. Configure Training Parameters
EPOCHS = 35        # Recommended epochs for high recall & mAP
BATCH_SIZE = 8     # Batch size optimized for Tesla T4 GPU
IMGSZ = 640        # Standard input resolution for YOLO & WebAssembly
MODEL_NAME = "trained_model.onnx"

In [ ]:
# 4. Process Dataset in Ephemeral RAM and Train
import json
import base64
import shutil
from pathlib import Path
import tempfile
import torch
from ultralytics import YOLO

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

temp_dir = Path(tempfile.mkdtemp(prefix="yolo_colab_"))

try:
    with open(dataset_filename, 'r', encoding='utf-8') as f:
        data = json.load(f)

    raw_images = data.get('images', data) if isinstance(data, dict) else data
    print(f"Total images: {len(raw_images)}")

    # Discover classes
    class_list = []
    for img in raw_images:
        for ann in img.get('annotations', []):
            lbl = ann.get('label') or ann.get('nomePeca') or 'OBJECT'
            lbl = lbl.strip()
            st = str(ann.get('status', '')).strip().upper()
            final_lbl = f"{lbl}_{st}" if st in ['OK', 'NOK'] and not lbl.upper().endswith(f"_{st}") else lbl
            if final_lbl not in class_list:
                class_list.append(final_lbl)

    if not class_list:
        class_list = ['OBJECT']

    class_to_id = {name: i for i, name in enumerate(class_list)}
    print(f"Detected classes: {class_list}")

    for split in ['train', 'val']:
        (temp_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (temp_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    # Decode images into ephemeral RAM disk
    for idx, img in enumerate(raw_images):
        durl = img.get('dataUrl') or ''
        if ',' not in durl: continue
        img_bytes = base64.b64decode(durl.split(',')[1])
        is_val = (idx % 7 == 0) and len(raw_images) >= 5
        split = 'val' if is_val else 'train'
        stem = f"frame_{idx:05d}"

        with open(temp_dir / 'images' / split / f"{stem}.jpg", 'wb') as fi:
            fi.write(img_bytes)

        lines = []
        for ann in img.get('annotations', []):
            lbl = ann.get('label') or 'OBJECT'
            st = str(ann.get('status', '')).strip().upper()
            flbl = f"{lbl}_{st}" if st in ['OK', 'NOK'] and not lbl.upper().endswith(f"_{st}") else lbl
            cid = class_to_id.get(flbl, 0)
            yolo = ann.get('yolo', {})
            xc = yolo.get('x_center', 0.5)
            yc = yolo.get('y_center', 0.5)
            w = yolo.get('width', 0.5)
            h = yolo.get('height', 0.5)
            lines.append(f"{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        with open(temp_dir / 'labels' / split / f"{stem}.txt", 'w') as fl:
            fl.write('\n'.join(lines) + '\n')

    # data.yaml
    yaml_text = f"path: '{temp_dir.as_posix()}'\ntrain: images/train\nval: images/val\nnames:\n"
    for i, name in enumerate(class_list):
        yaml_text += f"  {i}: '{name}'\n"

    with open(temp_dir / 'data.yaml', 'w') as fy:
        fy.write(yaml_text)

    # Train using YOLO26 nano architecture (C3k2 + C2PSA attention)
    try:
        model = YOLO('yolo11n.pt')
    except Exception:
        model = YOLO('yolov8n.pt')
    model.train(
        data=str(temp_dir / 'data.yaml'),
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMGSZ,
        device=0 if torch.cuda.is_available() else 'cpu',
        project=str(temp_dir / 'runs'),
        name='train_job',
        exist_ok=True,
        scale=0.6,
        degrees=15.0,
        translate=0.15,
        perspective=0.0005,
        shear=2.0,
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.4,
        cos_lr=True,
        close_mosaic=5
    )

    # Export to ONNX (opset=12 for full WebGL browser compatibility)
    best_pt = temp_dir / 'runs' / 'train_job' / 'weights' / 'best.pt'
    best_model = YOLO(str(best_pt))
    exported_onnx = best_model.export(format='onnx', imgsz=IMGSZ, opset=12, dynamic=False, simplify=True)

    # Patch WebGL Resize nodes and inject YOLO26 canonical metadata
    try:
        import onnx
        m = onnx.load(exported_onnx)
        for node in m.graph.node:
            if node.op_type == 'Resize':
                for attr in node.attribute:
                    if attr.name == 'mode':
                        attr.s = b'linear'
        meta_arch = m.metadata_props.add()
        meta_arch.key = 'architecture'
        meta_arch.value = 'yolo26'
        meta_fw = m.metadata_props.add()
        meta_fw.key = 'framework'
        meta_fw.value = 'YOLO26'
        onnx.checker.check_model(m)
        onnx.save(m, exported_onnx)
        print('✓ Applied WebGL linear resize patch & injected yolo26 metadata.')
    except Exception as e:
        print(f'Warning while patching ONNX: {e}')

    shutil.copy(exported_onnx, MODEL_NAME)
    print(f"\n✓ ONNX model successfully generated: {MODEL_NAME}")

finally:
    # Absolute Zero-Retention cleanup
    shutil.rmtree(temp_dir, ignore_errors=True)
    if os.path.exists(dataset_filename):
        os.remove(dataset_filename)
    print("🔒 Zero-Retention Policy Applied: All temporary images purged from memory.")

In [ ]:
# 5. Download the Compiled .ONNX Model to Your Machine
print("Starting download of the trained model...")
files.download(MODEL_NAME)